In [ ]:
import pandas as pd
import sys
sys.path.append("..")
from config import REGION_PROCESSED_DIR
from sklearn.model_selection import train_test_split   

# read the training table
train_df = pd.read_csv(REGION_PROCESSED_DIR / "training_table.csv")

# 1. correlation with the target
cols = ["past_crashes", "past_injured", "past_killed", "n_segments", "pci", "target_crashes"]
print(train_df[cols].corr()["target_crashes"])

# 2.  split 80% train, 20% test
train_set, test_set = train_test_split(train_df, test_size=0.2, random_state=42)
print(len(train_set), len(test_set))

past_crashes      0.780696
past_injured      0.763404
past_killed       0.271640
n_segments        0.420101
pci               0.066433
target_crashes    1.000000
Name: target_crashes, dtype: float64
6577 1645


In [ ]:
from sklearn.metrics import mean_absolute_error

# independent copy of the test set
test_set = test_set.copy()

# ---------- Prediction 1: past rate ----------
test_set["pred_rate"] = test_set["past_crashes"] * 3 / 18

mean_value = train_set["target_crashes"].mean()
test_set["pred_mean"] = mean_value

# ---------- MAE for each prediction ----------
y_true = test_set["target_crashes"]
print("MAE rate:", mean_absolute_error(y_true, test_set["pred_rate"]))
print("MAE mean:", mean_absolute_error(y_true, test_set["pred_mean"]))

# ---------- Top 10% coverage ----------
n = int(len(test_set) * 0.1)                       # size of top 10%
total_crashes = y_true.sum()                       # all test crashes

top_pred = test_set.sort_values("pred_rate", ascending=False).head(n)
coverage = top_pred["target_crashes"].sum() / total_crashes

top_ideal = test_set.sort_values("target_crashes", ascending=False).head(n)
ideal = top_ideal["target_crashes"].sum() / total_crashes

print("Top 10% coverage (model):", coverage)
print("Top 10% coverage (ideal):", ideal)

MAE rate: 0.5829787234042553
MAE mean: 1.0508378419221815
Top 10% coverage (model): 0.5286984640258691
Top 10% coverage (ideal): 0.6523848019401779


In [3]:
print("MAE zero:", mean_absolute_error(y_true, [0] * len(y_true)))

MAE zero: 0.7519756838905776
